In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import pandas as pd
df = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/Dữ liệu/student_data/shipments_realistic.csv")
df.shape

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


(566067, 22)

In [10]:
# ============================================================
# 1. IMPORT THƯ VIỆN
# ============================================================

import pandas as pd
import numpy as np
from IPython.display import display


# ============================================================
# 2. ĐỌC DỮ LIỆU GỐC
# ============================================================

file_path = (
    "/content/drive/MyDrive/Colab Notebooks/"
    "Dữ liệu/student_data/shipments_realistic.csv"
)

df = pd.read_csv(file_path)

print("Đọc dữ liệu gốc thành công!")
print("Kích thước dữ liệu gốc:", df.shape)

display(df.head())


# ============================================================
# 3. KIỂM TRA DỮ LIỆU GỐC
# ============================================================

print("\n========== KIỂM TRA DỮ LIỆU GỐC ==========")

print("\nThông tin dữ liệu:")
df.info()

print("\nSố lượng giá trị thiếu:")
print(df.isnull().sum())

print("\nSố dòng trùng hoàn toàn:")
print(df.duplicated().sum())

print("\nSố lượng shipper:")
print(df["shipper_id"].nunique())


# ============================================================
# 4. SAO CHÉP DỮ LIỆU
# ============================================================

df_clean = df.copy()


# ============================================================
# 5. CHUẨN HÓA TÊN CỘT
# ============================================================

df_clean.columns = (
    df_clean.columns
    .str.strip()
    .str.lower()
)


# ============================================================
# 6. CHUYỂN ĐỔI KIỂU DỮ LIỆU
# ============================================================

# Chuyển các cột ngày
df_clean["join_date"] = pd.to_datetime(
    df_clean["join_date"],
    errors="coerce"
)

# Chuyển các cột số nguyên
integer_columns = [
    "shipper_experience_years",
    "shipper_age"
]

for column in integer_columns:
    df_clean[column] = pd.to_numeric(
        df_clean[column],
        errors="coerce"
    )

# Chuyển các cột số thực
float_columns = [
    "shipper_rating",
    "delivery_success_rate",
    "average_delivery_time"
]

for column in float_columns:
    df_clean[column] = pd.to_numeric(
        df_clean[column],
        errors="coerce"
    )


# ============================================================
# 7. CHUẨN HÓA CÁC CỘT CHUỖI
# ============================================================

string_columns = [
    "shipper_id",
    "shipper_name",
    "shipper_phone",
    "shipper_company",
    "shipper_vehicle",
    "working_shift",
    "shipper_gender",
    "shipper_marital_status",
    "shipper_education"
]

for column in string_columns:
    df_clean[column] = (
        df_clean[column]
        .astype("string")
        .str.strip()
    )

# Đảm bảo số điện thoại được lưu dưới dạng chuỗi
df_clean["shipper_phone"] = (
    df_clean["shipper_phone"]
    .str.replace(r"\.0$", "", regex=True)
)


# ============================================================
# 8. KIỂM TRA VÀ LOẠI BỎ GIÁ TRỊ THIẾU
# ============================================================

required_columns = [
    "shipper_id",
    "shipper_name",
    "shipper_phone",
    "shipper_company",
    "shipper_experience_years",
    "shipper_rating",
    "delivery_success_rate",
    "average_delivery_time",
    "working_shift",
    "join_date",
    "shipper_gender",
    "shipper_age",
    "shipper_marital_status",
    "shipper_education"
]

print("\n========== GIÁ TRỊ THIẾU TRƯỚC KHI XỬ LÝ ==========")
print(df_clean[required_columns].isnull().sum())

df_clean = df_clean.dropna(
    subset=required_columns
).copy()

print("\nKích thước sau khi loại dòng thiếu:",
      df_clean.shape)


# ============================================================
# 9. KIỂM TRA GIÁ TRỊ KHÔNG HỢP LỆ
# ============================================================

# Kinh nghiệm không được âm
df_clean = df_clean[
    df_clean["shipper_experience_years"] >= 0
].copy()

# Tuổi nằm trong khoảng 18 đến 70
df_clean = df_clean[
    df_clean["shipper_age"].between(18, 70)
].copy()

# Đánh giá nằm trong khoảng 0 đến 5
df_clean = df_clean[
    df_clean["shipper_rating"].between(0, 5)
].copy()

# Tỷ lệ giao hàng thành công nằm trong khoảng 0 đến 100
df_clean = df_clean[
    df_clean["delivery_success_rate"].between(0, 100)
].copy()

# Thời gian giao hàng trung bình không được âm
df_clean = df_clean[
    df_clean["average_delivery_time"] >= 0
].copy()


# ============================================================
# 10. CHUYỂN CÁC CỘT SỐ NGUYÊN VỀ KIỂU INT
# ============================================================

df_clean["shipper_experience_years"] = (
    df_clean["shipper_experience_years"].astype(int)
)

df_clean["shipper_age"] = (
    df_clean["shipper_age"].astype(int)
)


# ============================================================
# 11. LOẠI BỎ DÒNG TRÙNG HOÀN TOÀN
# ============================================================

print("\nSố dòng trùng trước khi xóa:",
      df_clean.duplicated().sum())

df_clean = df_clean.drop_duplicates().copy()

print("Kích thước sau khi xóa dòng trùng:",
      df_clean.shape)


# ============================================================
# 12. TẠO CỘT DATE_OF_BIRTH ƯỚC TÍNH
# ============================================================

# Ngày tham chiếu để tính ngày sinh ước tính
reference_date = pd.Timestamp("2026-09-13")


def calculate_date_of_birth(age):
    return reference_date - pd.DateOffset(years=int(age))


df_clean["date_of_birth"] = df_clean["shipper_age"].apply(
    calculate_date_of_birth
)


# ============================================================
# 13. TẠO BẢNG SHIPPER
# ============================================================

shipper_columns = [
    "shipper_id",
    "shipper_name",
    "shipper_phone",
    "shipper_company",
    "date_of_birth",
    "join_date",
    "delivery_success_rate",
    "shipper_vehicle",
    "shipper_rating",
    "average_delivery_time",
    "working_shift",
    "shipper_gender",
    "shipper_marital_status",
    "shipper_education"
]

Shipper = df_clean[shipper_columns].copy()


# ============================================================
# 14. XỬ LÝ KHÓA CHÍNH SHIPPER_ID
# ============================================================

print("\n========== KIỂM TRA KHÓA CHÍNH ==========")

print("Số lượng shipper trước khi loại trùng:",
      Shipper["shipper_id"].nunique())

print("Số dòng trùng shipper_id:",
      Shipper["shipper_id"].duplicated().sum())

# Mỗi shipper chỉ xuất hiện một lần trong bảng SHIPPER
Shipper = Shipper.drop_duplicates(
    subset=["shipper_id"],
    keep="first"
).copy()


# ============================================================
# 15. SẮP XẾP DỮ LIỆU
# ============================================================

Shipper = Shipper.sort_values(
    by="shipper_id"
).reset_index(drop=True)


# ============================================================
# 16. KIỂM TRA BẢNG SHIPPER SAU KHI LÀM SẠCH
# ============================================================

print("\n========== KẾT QUẢ BẢNG SHIPPER ==========")

print("\nKích thước bảng:", Shipper.shape)

print("\nDanh sách cột:")
print(Shipper.columns.tolist())

print("\nSố lượng shipper:")
print(Shipper["shipper_id"].nunique())

print("\nGiá trị thiếu:")
print(Shipper.isnull().sum())

print("\nSố khóa chính bị trùng:")
print(Shipper["shipper_id"].duplicated().sum())

print("\nNăm dòng đầu tiên:")
display(Shipper.head())


# ============================================================
# 17. KIỂM TRA TÍNH HỢP LỆ
# ============================================================

assert Shipper["shipper_id"].is_unique
assert Shipper["shipper_id"].notna().all()

assert Shipper["date_of_birth"].notna().all()
assert Shipper["join_date"].notna().all()

assert Shipper["delivery_success_rate"].between(0, 100).all()
assert Shipper["shipper_rating"].between(0, 5).all()
assert Shipper["average_delivery_time"].ge(0).all()

print("\nDữ liệu SHIPPER đã vượt qua các kiểm tra cơ bản!")


# ============================================================
# 18. LƯU FILE CSV
# ============================================================

shipper_output_path = "/content/Shipper_clean.csv"

Shipper.to_csv(
    shipper_output_path,
    index=False,
    encoding="utf-8-sig"
)

print("\nĐã lưu file thành công:")
print(shipper_output_path)


# ============================================================
# 19. HIỂN THỊ TOÀN BỘ BẢNG SHIPPER
# ============================================================

display(Shipper)

Đọc dữ liệu gốc thành công!
Kích thước dữ liệu gốc: (566067, 22)


,shipper_id,order_id,ship_date,delivery_date,shipping_fee,shipper_company,shipper_vehicle,shipper_experience_years,shipper_rating,delivery_success_rate,...,join_date,shipper_name,shipper_phone,shipper_gender,shipper_age,shipper_marital_status,shipper_education,city,region,district
0,SHP00001,1,2012-07-07,2012-07-11,1.37,Viettel Post,Truck,7,5.0,99.0,...,2026-03-17,Bùi Văn Long,991476209,Male,27,Married,Bachelor,Phan Rang-Thap Cham,Central,District #25
1,SHP00002,2,2012-07-06,2012-07-10,2.60,J&T Express,Van,2,4.9,98.4,...,2025-01-29,Trần Anh Khánh,959297982,Male,41,Married,Bachelor,Phan Thiet,Central,District #29
2,SHP00003,3,2012-07-04,2012-07-07,2.38,GHN,Motorbike,10,4.8,95.1,...,2019-11-13,Hoàng Thị Khánh,927142576,Male,30,Single,High School,Long Xuyen,West,District #34
3,SHP00004,4,2012-07-05,2012-07-11,2.49,Viettel Post,Truck,8,5.0,96.3,...,2025-12-22,Trần Đức Vy,971617475,Female,42,Married,College,Kon Tum,Central,District #27
4,SHP00005,6,2012-07-09,2012-07-16,25.79,BEST Express,Truck,10,4.6,95.7,...,2019-12-19,Trần Minh Cường,979196342,Male,31,Married,College,Da Nang,Central,District #23



========== KIỂM TRA DỮ LIỆU GỐC ==========

Thông tin dữ liệu:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 566067 entries, 0 to 566066
Data columns (total 22 columns):
 #   Column                    Non-Null Count   Dtype  
---  ------                    --------------   -----  
 0   shipper_id                566067 non-null  object 
 1   order_id                  566067 non-null  int64  
 2   ship_date                 566067 non-null  object 
 3   delivery_date             566067 non-null  object 
 4   shipping_fee              566067 non-null  float64
 5   shipper_company           566067 non-null  object 
 6   shipper_vehicle           566067 non-null  object 
 7   shipper_experience_years  566067 non-null  int64  
 8   shipper_rating            566067 non-null  float64
 9   delivery_success_rate     566067 non-null  float64
 10  average_delivery_time     566067 non-null  int64  
 11  working_shift             566067 non-null  object 
 12  join_date                 566067 non

,shipper_id,shipper_name,shipper_phone,shipper_company,date_of_birth,join_date,delivery_success_rate,shipper_vehicle,shipper_rating,average_delivery_time,working_shift,shipper_gender,shipper_marital_status,shipper_education
0,SHP00001,Bùi Văn Long,991476209,Viettel Post,1999-09-13,2026-03-17,99.0,Truck,5.0,61,Evening,Male,Married,Bachelor
1,SHP00002,Trần Anh Khánh,959297982,J&T Express,1985-09-13,2025-01-29,98.4,Van,4.9,72,Afternoon,Male,Married,Bachelor
2,SHP00003,Hoàng Thị Khánh,927142576,GHN,1996-09-13,2019-11-13,95.1,Motorbike,4.8,53,Evening,Male,Single,High School
3,SHP00004,Trần Đức Vy,971617475,Viettel Post,1984-09-13,2025-12-22,96.3,Truck,5.0,53,Evening,Female,Married,College
4,SHP00005,Trần Minh Cường,979196342,BEST Express,1995-09-13,2019-12-19,95.7,Truck,4.6,62,Morning,Male,Married,College



Dữ liệu SHIPPER đã vượt qua các kiểm tra cơ bản!

Đã lưu file thành công:
/content/Shipper_clean.csv


,shipper_id,shipper_name,shipper_phone,shipper_company,date_of_birth,join_date,delivery_success_rate,shipper_vehicle,shipper_rating,average_delivery_time,working_shift,shipper_gender,shipper_marital_status,shipper_education
0,SHP00001,Bùi Văn Long,991476209,Viettel Post,1999-09-13,2026-03-17,99.0,Truck,5.0,61,Evening,Male,Married,Bachelor
1,SHP00002,Trần Anh Khánh,959297982,J&T Express,1985-09-13,2025-01-29,98.4,Van,4.9,72,Afternoon,Male,Married,Bachelor
2,SHP00003,Hoàng Thị Khánh,927142576,GHN,1996-09-13,2019-11-13,95.1,Motorbike,4.8,53,Evening,Male,Single,High School
3,SHP00004,Trần Đức Vy,971617475,Viettel Post,1984-09-13,2025-12-22,96.3,Truck,5.0,53,Evening,Female,Married,College
4,SHP00005,Trần Minh Cường,979196342,BEST Express,1995-09-13,2019-12-19,95.7,Truck,4.6,62,Morning,Male,Married,College
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
75,SHP00076,Trần Thanh Long,933140008,BEST Express,1997-09-13,2020-09-25,95.1,Truck,4.7,65,Afternoon,Male,Married,Bachelor
76,SHP00077,Trần Văn Giang,931398461,BEST Express,1988-09-13,2019-11-11,99.4,Truck,4.7,41,Morning,Female,Married,Bachelor
77,SHP00078,Đặng Văn Bình,999125359,Shopee Express,1997-09-13,2025-09-25,96.8,Motorbike,5.0,38,Evening,Female,Single,High School
78,SHP00079,Đặng Thanh An,993275153,Shopee Express,1980-09-13,2018-09-09,95.9,Motorbike,4.5,68,Morning,Male,Married,Bachelor
